# Geocoding & Standardisasi Kandidat Kantor BUMN/Pemerintah

**Tujuan notebook ini:**
Mengubah daftar nama + alamat kantor BUMN/pemerintah di Kota Bandung menjadi
data spasial (koordinat) yang formatnya sama dengan `candidates_J.csv`
(Himpunan J) yang sudah ada, sehingga bisa digabung sebagai kandidat
tambahan klaster Kantor.

**Alur:**
1. Baca daftar nama + alamat mentah
2. Geocode tiap alamat jadi koordinat (lon, lat) pakai Nominatim (OpenStreetMap)
3. Snap ke node jalan terdekat pada graf jaringan jalan
4. Filter wilayah -- buang yang ternyata di luar Kota Bandung
5. Simpan dalam format standar: `nama, klaster, lon, lat, nearest_node, dist_to_node_m`

**Catatan penting:**
- Nominatim (geocoder gratis) punya batas 1 permintaan/detik -- notebook ini
  otomatis jeda ±1.1 detik antar alamat, jadi wajar prosesnya makan waktu
  (44 alamat &asymp; 1 menit).
- Sebagian alamat berpotensi **gagal** di-geocode otomatis (alamat kurang
  presisi). Yang gagal akan ditampilkan di akhir, supaya bisa dicari
  koordinatnya manual lewat Google Maps.
- Kolom `confidence` pada data mentah menandai seberapa presisi alamat
  sumbernya -- hasil dengan confidence `rendah` sebaiknya dicek ulang manual
  sebelum dipakai sebagai kandidat final.
- **Data lama (Himpunan I, Himpunan J yang sudah ada) tidak disentuh sama
  sekali oleh notebook ini** -- hasilnya disimpan sebagai file terpisah.

## 1. Import library

In [34]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # sesuaikan kalau struktur foldernya beda
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
UTM_EPSG = 32748
KLASTER_NAME = "C_KantorBUMN"


## 2. Muat daftar alamat mentah

File `bumn_candidates_raw.csv` berisi 44 kantor BUMN/pemerintah di Kota
Bandung, hasil gabungan dari 2 sumber (lawyersclubs.com dan alamatelpon.com),
sudah dibuang entri yang jelas tidak valid (misalnya perusahaan yang sudah
bubar, atau alamat yang ternyata di luar Kota Bandung).

In [35]:
df = pd.read_csv(RAW_DIR / "bumn_candidates_raw.csv")
print(f"Total alamat: {len(df)}")
df.head()

Total alamat: 44


,nama,alamat,confidence,catatan
0,PT Dirgantara Indonesia (IPTN),"Jl. Pajajaran No. 154, Husen Sastranegara, Cic...",tinggi,NaN
1,PT Garuda Indonesia,"Jl. Asia Afrika No. 81, Paledang, Braga, Sumur...",tinggi,NaN
2,PT Len Industri,"Jl. Soekarno Hatta No. 442, Pasirluyu, Regol, ...",tinggi,NaN
3,PT Kereta Api Indonesia,"Jl. Perintis Kemerdekaan No. 1, Babakan Ciamis...",tinggi,NaN
4,PT Telkom Bandung,"Jl. Bengawan No. 81, Cihapit, Bandung Wetan, K...",tinggi,NaN


## 3. Geocoding alamat &rarr; koordinat

Tiap alamat dikirim ke Nominatim (geocoder gratis dari OpenStreetMap) untuk
dicari koordinat lon/lat-nya.

**Strategi bertingkat (3 level)** -- Nominatim itu cukup kaku, alamat
dengan nomor rumah presisi seringkali tidak ditemukan walau jalannya ada
di peta. Jadi tiap alamat dicoba beberapa cara, dari yang paling presisi
ke yang paling umum, berhenti begitu salah satu berhasil:
1. **Alamat lengkap** apa adanya
2. **Tanpa nomor rumah** (misal "Jl. Asia Afrika No. 107" &rarr; "Jl. Asia Afrika")
3. **Nama tempat + kota** (manfaatkan kalau tempatnya sendiri sudah
   terdaftar sebagai POI di OpenStreetMap, terlepas dari alamat jalannya)

In [36]:
import re

def bikin_query_bertingkat(nama, alamat):
    # level 1: alamat lengkap
    level1 = f"{alamat}, Indonesia"
    # level 2: alamat tanpa nomor rumah (No. 107, No 107, no.107, No. 57-59, dst)
    alamat_tanpa_nomor = re.sub(r",?\s*No\.?\s*\d+[A-Za-z]?(\s*-\s*\d+[A-Za-z]?)?", "", alamat, flags=re.IGNORECASE)
    level2 = f"{alamat_tanpa_nomor}, Indonesia"
    # level 3: nama tempat + kota (manfaatkan POI di OSM)
    kota = "Kota Bandung, Jawa Barat, Indonesia"
    level3 = f"{nama}, {kota}"
    # buang duplikat kalau level2 == level1 (misal alamat tidak ada nomor rumah)
    queries = [level1]
    if level2 != level1:
        queries.append(level2)
    queries.append(level3)
    return queries


geolocator = Nominatim(user_agent="riset_spklu_bandung_magang")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

lats, lons, status, level_berhasil = [], [], [], []
for i, row in df.iterrows():
    queries = bikin_query_bertingkat(row["nama"], row["alamat"])
    location = None
    level_ke = None
    for lvl, q in enumerate(queries, start=1):
        try:
            location = geocode(q)
        except Exception as e:
            print(f"    (error jaringan di level {lvl}: {e})")
            location = None
        if location:
            level_ke = lvl
            break

    if location:
        lats.append(location.latitude)
        lons.append(location.longitude)
        status.append("berhasil")
        level_berhasil.append(level_ke)
    else:
        lats.append(None)
        lons.append(None)
        status.append("GAGAL")
        level_berhasil.append(None)
    print(f"[{i+1}/{len(df)}] {row['nama']}: {status[-1]}"
          + (f" (level {level_ke})" if level_ke else ""))

df["lat"] = lats
df["lon"] = lons
df["geocode_status"] = status
df["geocode_level"] = level_berhasil

[1/44] PT Dirgantara Indonesia (IPTN): berhasil (level 2)
[2/44] PT Garuda Indonesia: GAGAL
[3/44] PT Len Industri: berhasil (level 2)
[4/44] PT Kereta Api Indonesia: berhasil (level 2)
[5/44] PT Telkom Bandung: berhasil (level 2)
[6/44] PT INTI: berhasil (level 3)
[7/44] PT Perusahaan Gas Negara Rayon Bandung: berhasil (level 2)
[8/44] PT Sucofindo: berhasil (level 2)
[9/44] PT PLN UID Jawa Barat: berhasil (level 2)
[10/44] PT Biofarma: berhasil (level 2)
[11/44] PT Kimia Farma: berhasil (level 2)
[12/44] PT Pertamina Bandung: berhasil (level 1)
[13/44] PT Pindad: berhasil (level 3)
[14/44] PT Indah Karya: GAGAL
[15/44] PT Adhi Karya: berhasil (level 2)
[16/44] PT Waskita Karya: berhasil (level 2)
[17/44] PT PPRO BIJB Aerocity Development: berhasil (level 2)
[18/44] PT Asuransi Jiwasraya Kanwil Bandung: berhasil (level 2)
[19/44] PT Bhanda Ghara Reksa: berhasil (level 2)
[20/44] RSUP Dr. Hasan Sadikin (RSHS): berhasil (level 2)
[21/44] Bank Mandiri KC Asia Afrika: berhasil (level 2)
[

### Cek alamat yang gagal di-geocode otomatis

In [37]:
gagal = df[df["geocode_status"] == "GAGAL"]
print(f"{len(gagal)} alamat gagal di-geocode otomatis:")
gagal[["nama", "alamat"]]

4 alamat gagal di-geocode otomatis:


,nama,alamat
1,PT Garuda Indonesia,"Jl. Asia Afrika No. 81, Paledang, Braga, Sumur..."
13,PT Indah Karya,"Jl. Golf Raya, Ujungberung, Kota Bandung, Jawa..."
28,PT Jamsostek/BPJS Ketenagakerjaan,"Jl. PHH Mustofa No. 39, Neglasari, Cibeunying ..."
30,PT Taspen,"Jl. PHH Mustopa No. 78, Cikutra, Cibeunying Ki..."


> Kalau ada yang gagal, cari koordinatnya manual lewat Google Maps
> (klik-tahan lokasi &rarr; copy koordinat), lalu isi manual di cell berikut
> (opsional, lewati kalau tidak ada yang gagal).

In [ ]:
# Contoh cara isi manual kalau ada yang gagal geocoding:
# df.loc[df["nama"] == "Nama Kantor X", ["lat", "lon"]] = [-6.9123, 107.6123]

df_ok = df.dropna(subset=["lat", "lon"]).copy()
print(f"Siap diproses lebih lanjut: {len(df_ok)} dari {len(df)}")

Siap diproses lebih lanjut: 40 dari 44


### Perbaiki titik yang koordinatnya bentrok persis

Beberapa kandidat bisa berakhir di koordinat yang **persis sama** -- ini
terjadi saat 2+ kantor berada di jalan yang sama, tapi Nominatim gagal
menemukan nomor gedung spesifiknya sehingga jatuh ke titik referensi jalan
yang sama (misalnya PLN dan BRI yang sama-sama di Jl. Asia Afrika).

Karena kita **tahu** titik-titik ini memang berada di jalan yang sama
(hanya tidak tahu persis nomor gedungnya), diberikan pergeseran kecil
acak (50-150 meter, arah acak) -- bukan tebakan sembarangan, melainkan
representasi yang lebih realistis daripada membiarkan beberapa kantor
"menumpuk" di satu titik identik. Random seed tetap supaya hasilnya
konsisten setiap notebook dijalankan ulang.

**Ditempatkan di sini (sebelum snap ke jalan)** supaya `nearest_node`
yang dihitung nanti konsisten dengan posisi barunya, bukan posisi lama.

In [39]:
import numpy as np

np.random.seed(42)  # reproducible

df_ok["_coord_key"] = df_ok["lat"].round(6).astype(str) + "_" + df_ok["lon"].round(6).astype(str)
grup_bentrok = df_ok[df_ok.duplicated(subset="_coord_key", keep=False)]["_coord_key"].unique()

print(f"{len(grup_bentrok)} grup koordinat bentrok ditemukan "
      f"({df_ok['_coord_key'].isin(grup_bentrok).sum()} titik total).")

for key in grup_bentrok:
    idx = df_ok[df_ok["_coord_key"] == key].index
    print(f"  Grup: {df_ok.loc[idx, 'nama'].tolist()}")
    for i in idx:
        # geser 50-150 meter ke arah acak (dikonversi ke derajat lat/lon)
        radius_m = np.random.uniform(50, 150)
        sudut = np.random.uniform(0, 2 * np.pi)
        d_lat = (radius_m * np.cos(sudut)) / 111_000  # 1 derajat lat ~ 111km
        d_lon = (radius_m * np.sin(sudut)) / (111_000 * np.cos(np.radians(df_ok.loc[i, "lat"])))
        df_ok.loc[i, "lat"] += d_lat
        df_ok.loc[i, "lon"] += d_lon

df_ok = df_ok.drop(columns="_coord_key")

cek_ulang = df_ok.duplicated(subset=["lat", "lon"]).sum()
print(f"\nSisa titik bentrok persis setelah perbaikan: {cek_ulang}")

6 grup koordinat bentrok ditemukan (14 titik total).
  Grup: ['PT PLN UID Jawa Barat', 'PT Asuransi Jiwasraya Kanwil Bandung', 'PT Bank Rakyat Indonesia (BRI)']
  Grup: ['PT Biofarma', 'RSUP Dr. Hasan Sadikin (RSHS)']
  Grup: ['Bank Mandiri KC Asia Afrika', 'PT Pos Indonesia']
  Grup: ['Stasiun Bandung (KAI)', 'Perum DAMRI']
  Grup: ['PT Jasa Raharja', 'Perum Bulog', 'Perum Perhutani']
  Grup: ['PD Kebersihan', 'Perum Pembangunan Perumahan Nasional (Perumnas)']

Sisa titik bentrok persis setelah perbaikan: 0


## 4. Muat graf jaringan jalan

Menggunakan graf yang sudah dibuat sebelumnya di Tahap 2 (tidak diunduh ulang).

In [40]:
G = ox.load_graphml(PROCESSED_DIR / "bandung_drive_utm48s.graphml")
print(f"{len(G.nodes)} node, {len(G.edges)} edge")

25547 node, 59040 edge


## 5. Snap tiap kandidat ke node jalan terdekat

In [41]:
gdf = gpd.GeoDataFrame(
    df_ok, geometry=gpd.points_from_xy(df_ok["lon"], df_ok["lat"]), crs="EPSG:4326"
).to_crs(f"EPSG:{UTM_EPSG}")

nearest_nodes, dists = ox.distance.nearest_nodes(
    G, gdf.geometry.x, gdf.geometry.y, return_dist=True
)
gdf["nearest_node"] = nearest_nodes
gdf["dist_to_node_m"] = dists
gdf[["nama", "nearest_node", "dist_to_node_m"]].head()

,nama,nearest_node,dist_to_node_m
0,PT Dirgantara Indonesia (IPTN),25433949,44.308734
2,PT Len Industri,6619012655,126.841925
3,PT Kereta Api Indonesia,29354164,31.821984
4,PT Telkom Bandung,11367059254,115.050941
5,PT INTI,5355955211,94.328740


## 6. Filter wilayah -- buang yang di luar Kota Bandung

Menggunakan batas 151 polygon kelurahan yang sudah didownload sebelumnya
(`kelurahan_bandung.json`), sama seperti proses filter wilayah pada
Himpunan I dan ground truth SPKLU.

In [42]:
kelurahan_cache = RAW_DIR / "kelurahan_bandung.json"

if kelurahan_cache.exists():
    kelurahan = gpd.read_file(kelurahan_cache).set_crs("EPSG:4326", allow_override=True)
    boundary = kelurahan.to_crs(f"EPSG:{UTM_EPSG}").geometry.unary_union

    before = len(gdf)
    di_dalam = gdf.geometry.within(boundary)
    di_luar = gdf[~di_dalam]

    if len(di_luar) > 0:
        print(f"{len(di_luar)} dibuang (di luar Kota Bandung):")
        print(di_luar["nama"].tolist())

    gdf = gdf[di_dalam].reset_index(drop=True)
    print(f"\nSisa: {len(gdf)} dari {before}")
else:
    print("File batas kelurahan tidak ditemukan -- filter wilayah dilewati!")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_44228\3930368282.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  boundary = kelurahan.to_crs(f"EPSG:{UTM_EPSG}").geometry.unary_union



Sisa: 40 dari 40


## 7. Simpan hasil dengan format standar

Format kolomnya disamakan dengan `candidates_J.csv` yang sudah ada, supaya
nanti tinggal digabung.

In [43]:
gdf["klaster"] = KLASTER_NAME
gdf_wgs84 = gdf.to_crs("EPSG:4326")

out = gdf_wgs84[["nama", "klaster", "nearest_node", "dist_to_node_m"]].copy()
out["lon"] = gdf_wgs84.geometry.x
out["lat"] = gdf_wgs84.geometry.y

out_path = PROCESSED_DIR / "candidates_kantor_bumn.csv"
out.to_csv(out_path, index=False)

print(f"Selesai. {len(out)} kandidat tersimpan di: {out_path}")
out

Selesai. 40 kandidat tersimpan di: d:\Magang\Week 1\spklu_bandung\data\processed\candidates_kantor_bumn.csv


,nama,klaster,nearest_node,dist_to_node_m,lon,lat
0,PT Dirgantara Indonesia (IPTN),C_KantorBUMN,25433949,44.308734,107.583220,-6.905137
1,PT Len Industri,C_KantorBUMN,6619012655,126.841925,107.616052,-6.948875
2,PT Kereta Api Indonesia,C_KantorBUMN,29354164,31.821984,107.607465,-6.914611
3,PT Telkom Bandung,C_KantorBUMN,11367059254,115.050941,107.628853,-6.910346
4,PT INTI,C_KantorBUMN,5355955211,94.328740,107.607060,-6.938626
5,PT Perusahaan Gas Negara Rayon Bandung,C_KantorBUMN,1849960039,16.402530,107.639413,-6.918909
6,PT Sucofindo,C_KantorBUMN,2325453514,61.630752,107.585851,-6.944066
7,PT PLN UID Jawa Barat,C_KantorBUMN,5358212335,92.535810,107.610495,-6.920832
8,PT Biofarma,C_KantorBUMN,5318313305,31.975453,107.600155,-6.882260
9,PT Kimia Farma,C_KantorBUMN,5334388827,33.556051,107.604168,-6.909860


## 8. (Opsional) Gabung dengan Himpunan J yang sudah ada

Cell ini **tidak menimpa** `candidates_J.csv` yang lama -- hasil gabungan
disimpan sebagai file baru, biar data lama tetap aman.

In [45]:
candidates_lama = pd.read_csv(PROCESSED_DIR / "candidates_J.csv")
gabungan = pd.concat([candidates_lama, out], ignore_index=True)

gabungan.to_csv(PROCESSED_DIR / "candidates_J_plus_bumn.csv", index=False)
print(f"Total kandidat gabungan: {len(gabungan)}")
gabungan["klaster"].value_counts()

Total kandidat gabungan: 130


klaster
A_SPBU          65
C_KantorBUMN    40
B_Mall          25
Name: count, dtype: int64